In [1]:
import os
import re
import csv
import glob
import random
import threading
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
!pip install opencv-python
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------- CONFIGURATION ----------------
base_folder = r"E:\planetscope_lake_ice\Data\Input"
study_sites = ["YKD_FullCloud", "NS_FullCloud", "YF_FullCloud"]
samples_per_season = 100  # Sample 100 images per season per site
max_display_side = 1200
downsample_step = 2
preload_count = 2
# ------------------------------------------------


def load_multiband_image(path):
    """Load image with proper multi-band support - tries rasterio first, then GDAL."""
    
    # Try rasterio first (best for multi-band GeoTIFFs)
    try:
        import rasterio
        with rasterio.open(path) as src:
            # Read all bands
            bands = []
            for i in range(1, src.count + 1):
                bands.append(src.read(i))
            
            if len(bands) == 0:
                return None
            elif len(bands) == 1:
                return bands[0]
            else:
                # Stack bands: (bands, height, width) -> (height, width, bands)
                img = np.stack(bands, axis=-1)
                return img
    except ImportError:
        pass
    except Exception as e:
        pass
    
    # Fallback to GDAL
    try:
        from osgeo import gdal
        gdal.UseExceptions()
        ds = gdal.Open(path)
        if ds is None:
            return None
        
        bands = []
        for i in range(1, ds.RasterCount + 1):
            band = ds.GetRasterBand(i)
            bands.append(band.ReadAsArray())
        
        ds = None
        
        if len(bands) == 0:
            return None
        elif len(bands) == 1:
            return bands[0]
        else:
            img = np.stack(bands, axis=-1)
            return img
    except ImportError:
        print("ERROR: Neither rasterio nor GDAL available. Install with: pip install rasterio")
        return None
    except Exception as e:
        return None


def create_rgb_display(img, is_freezeup=False):
    """
    Create RGB display from multi-band Planet image.
    Planet band order: B(0), G(1), R(2), NIR(3) for 4-band
    """
    if img is None:
        return None
    
    # Handle single band images
    if img.ndim == 2:
        img = np.repeat(img[:, :, np.newaxis], 3, axis=2)
    
    # Downsample for faster display
    img_down = img[::downsample_step, ::downsample_step].astype(np.float32)
    
    # Extract RGB bands
    num_bands = img_down.shape[2] if img_down.ndim == 3 else 1
    
    if num_bands >= 3:
        # Planet order is BGR (or BGRN for 4-band)
        # For OpenCV display (BGR format), we keep BGR order
        b_band = img_down[:, :, 0]  # Blue
        g_band = img_down[:, :, 1]  # Green
        r_band = img_down[:, :, 2]  # Red
        
        rgb = np.dstack([b_band, g_band, r_band])
    elif num_bands == 1:
        rgb = np.dstack([img_down[:, :, 0], img_down[:, :, 0], img_down[:, :, 0]])
    else:
        return None
    
    # Normalize from 16-bit to 0-1 range
    if rgb.max() > 1.5:
        rgb = rgb / 10000.0
    
    rgb = np.clip(rgb, 0, 1)
    
    # Apply different processing based on scene type
    if is_freezeup:
        # Aggressive contrast stretching for bright snow/ice scenes
        rgb_stretched = np.zeros_like(rgb)
        for i in range(3):
            band = rgb[:, :, i]
            p1, p99 = np.percentile(band[band > 0], (1, 99))
            if p99 > p1:
                band_stretched = (band - p1) / (p99 - p1)
            else:
                band_stretched = band
            rgb_stretched[:, :, i] = np.clip(band_stretched, 0, 1)
        
        rgb_stretched = np.power(rgb_stretched, 0.7)
        
        hsv = cv2.cvtColor((rgb_stretched * 255).astype(np.uint8), cv2.COLOR_BGR2HSV)
        hsv[:, :, 1] = np.clip(hsv[:, :, 1] * 1.2, 0, 255).astype(np.uint8)
        rgb_final = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR).astype(np.float32) / 255.0
    else:
        # Standard contrast stretch for breakup scenes
        rgb_stretched = np.zeros_like(rgb)
        for i in range(3):
            band = rgb[:, :, i]
            p2, p98 = np.percentile(band[band > 0], (2, 98))
            if p98 > p2:
                band_stretched = (band - p2) / (p98 - p2)
            else:
                band_stretched = band
            rgb_stretched[:, :, i] = np.clip(band_stretched, 0, 1)
        
        rgb_final = rgb_stretched
    
    # Convert to 8-bit for display
    rgb_8bit = (rgb_final * 255).astype(np.uint8)
    
    return rgb_8bit


def quick_rgb(image_path):
    """Load and process image based on parent folder."""
    is_freezeup = bool(re.search("freezeup", image_path, re.IGNORECASE))
    img = load_multiband_image(image_path)
    return create_rgb_display(img, is_freezeup)


def create_display_image(img, index, total, img_info):
    """Create final display image with info overlay."""
    if img is None:
        return None
    
    h, w = img.shape[:2]
    scale = min(max_display_side / max(h, w), 1.0)
    display = cv2.resize(img, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    
    # Create semi-transparent overlay for info
    overlay = display.copy()
    info_height = 100
    cv2.rectangle(overlay, (0, 0), (display.shape[1], info_height), (40, 40, 40), -1)
    cv2.addWeighted(overlay, 0.7, display, 0.3, 0, display)
    
    # Add counter
    counter_text = f"{index + 1}/{total}"
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(display, counter_text, (15, 30), font, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
    
    # Add site and season info
    site_season = f"{img_info['site']} - {img_info['season']} {img_info['year']}"
    cv2.putText(display, site_season, (15, 55), font, 0.6, (200, 200, 200), 1, cv2.LINE_AA)
    
    # Add filename (truncated if too long)
    max_name_len = 50
    img_name = img_info['filename']
    display_name = img_name if len(img_name) <= max_name_len else img_name[:max_name_len-3] + "..."
    cv2.putText(display, display_name, (15, 80), font, 0.5, (180, 180, 180), 1, cv2.LINE_AA)
    
    # Add instructions at bottom (2 lines)
    instr_y1 = display.shape[0] - 30
    instr_y2 = display.shape[0] - 10
    cv2.putText(display, "X=No Clouds  |  C=Cloudy  |  V=Very Cloudy", 
                (15, instr_y1), font, 0.6, (255, 255, 0), 1, cv2.LINE_AA)
    cv2.putText(display, "B=Back  |  S=Skip  |  Q=Quit", 
                (15, instr_y2), font, 0.6, (255, 255, 0), 1, cv2.LINE_AA)
    
    # Add progress bar
    progress = (index + 1) / total
    bar_width = display.shape[1] - 30
    bar_height = 8
    bar_y = info_height - bar_height - 8
    cv2.rectangle(display, (15, bar_y), (15 + int(bar_width * progress), bar_y + bar_height), 
                  (0, 255, 0), -1)
    cv2.rectangle(display, (15, bar_y), (15 + bar_width, bar_y + bar_height), 
                  (100, 100, 100), 1)
    
    return display


class ImagePreloader:
    """Preload images in background for faster navigation."""
    def __init__(self, max_workers=2):
        self.executor = ThreadPoolExecutor(max_workers=max_workers)
        self.cache = {}
        self.futures = {}
    
    def preload(self, paths):
        """Start preloading images."""
        for path in paths:
            if path not in self.cache and path not in self.futures:
                future = self.executor.submit(quick_rgb, path)
                self.futures[path] = future
    
    def get(self, path):
        """Get image from cache or wait for it to load."""
        if path in self.cache:
            return self.cache.pop(path)
        
        if path in self.futures:
            img = self.futures.pop(path).result()
            return img
        
        return quick_rgb(path)
    
    def cleanup(self):
        """Cancel pending futures and shutdown."""
        for future in self.futures.values():
            future.cancel()
        self.executor.shutdown(wait=False)


def collect_images_by_site_season():
    """
    Collect and randomly sample images for each study site and season.
    Returns dict with structure: {site: {season: [image_paths]}}
    """
    site_season_images = {}
    
    for site in study_sites:
        site_path = os.path.join(base_folder, site)
        if not os.path.exists(site_path):
            print(f"Warning: Site folder not found: {site_path}")
            continue
        
        site_season_images[site] = {'Breakup': [], 'Freezeup': []}
        
        # Find all season folders (Breakup_YYYY and Freezeup_YYYY)
        season_folders = glob.glob(os.path.join(site_path, "*_*"))
        
        # Group by season
        breakup_images = []
        freezeup_images = []
        
        for folder in season_folders:
            folder_name = os.path.basename(folder)
            
            # Determine season
            if folder_name.startswith("Breakup_"):
                season = "Breakup"
            elif folder_name.startswith("Freezeup_"):
                season = "Freezeup"
            else:
                continue
            
            # Find all SR TIF files (exclude udm files)
            tif_files = glob.glob(os.path.join(folder, "*.tif"))
            sr_files = [
                f for f in tif_files 
                if "udm" not in os.path.basename(f).lower()
            ]
            
            if season == "Breakup":
                breakup_images.extend(sr_files)
            else:
                freezeup_images.extend(sr_files)
        
        # Randomly sample up to samples_per_season images from each season
        if breakup_images:
            sample_size = min(samples_per_season, len(breakup_images))
            site_season_images[site]['Breakup'] = random.sample(breakup_images, sample_size)
            print(f"{site} - Breakup: Sampled {sample_size} from {len(breakup_images)} images")
        
        if freezeup_images:
            sample_size = min(samples_per_season, len(freezeup_images))
            site_season_images[site]['Freezeup'] = random.sample(freezeup_images, sample_size)
            print(f"{site} - Freezeup: Sampled {sample_size} from {len(freezeup_images)} images")
    
    return site_season_images


def extract_image_info(image_path):
    """Extract site, season, year, and filename from image path."""
    parts = Path(image_path).parts
    filename = os.path.basename(image_path)
    
    # Find site
    site = None
    for s in study_sites:
        if s in parts:
            site = s
            break
    
    # Find season and year from folder name (e.g., "Breakup_2019")
    season = "Unknown"
    year = "Unknown"
    for part in parts:
        if "_" in part:
            match = re.match(r"(Breakup|Freezeup)_(\d{4})", part)
            if match:
                season = match.group(1)
                year = match.group(2)
                break
    
    return {
        'site': site or "Unknown",
        'season': season,
        'year': year,
        'filename': filename,
        'full_path': image_path
    }


def main():
    print("=" * 60)
    print("PLANET IMAGE CLOUD LABELER - Random Sample Mode")
    print("=" * 60)
    print(f"\nSampling {samples_per_season} images per season per site...")
    print(f"Study sites: {', '.join(study_sites)}\n")
    
    # Collect and sample images
    site_season_images = collect_images_by_site_season()
    
    # Flatten into a single list with metadata
    all_images = []
    for site, seasons in site_season_images.items():
        for season, images in seasons.items():
            for img_path in images:
                img_info = extract_image_info(img_path)
                all_images.append(img_info)
    
    if not all_images:
        print("No images found to label!")
        return
    
    print(f"\nTotal images to label: {len(all_images)}")
    print("=" * 60)
    
    # Prepare output files for each site
    output_files = {}
    csv_writers = {}
    results_by_site = {}
    
    for site in study_sites:
        site_path = os.path.join(base_folder, site)
        if os.path.exists(site_path):
            output_csv = os.path.join(site_path, "cloud_analysis_output.csv")
            output_files[site] = output_csv
            results_by_site[site] = {}
            
            # Load existing labels if available
            if os.path.exists(output_csv) and os.path.getsize(output_csv) > 0:
                try:
                    existing = pd.read_csv(output_csv)
                    if "image" in existing.columns and "label" in existing.columns:
                        results_by_site[site] = dict(zip(existing["image"], existing["label"]))
                        print(f"Loaded {len(results_by_site[site])} existing labels for {site}")
                except Exception as e:
                    print(f"Warning: Could not load existing labels for {site}: {e}")
            
            # Open CSV for writing
            csvfile = open(output_csv, "w", newline="", encoding="utf-8")
            writer = csv.writer(csvfile)
            writer.writerow(["site", "season", "year", "image", "label"])
            csv_writers[site] = (csvfile, writer)
    
    # Create window
    window_name = "Planet Image Cloud Labeler"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(window_name, max_display_side, max_display_side)
    
    # Initialize preloader
    preloader = ImagePreloader(max_workers=preload_count)
    
    # Track current index for back functionality
    current_idx = 0
    labeled_history = []  # Track (idx, img_info, label) for undo
    
    try:
        while current_idx < len(all_images):
            idx = current_idx
            img_info = all_images[idx]
            img_path = img_info['full_path']
            img_name = img_info['filename']
            site = img_info['site']
            
            # Skip already labeled images
            if site in results_by_site and img_name in results_by_site[site]:
                print(f"Skipping already labeled: {img_name}")
                current_idx += 1
                continue
            
            # Preload upcoming images
            preload_paths = [
                all_images[i]['full_path']
                for i in range(idx + 1, min(idx + 1 + preload_count, len(all_images)))
            ]
            preloader.preload(preload_paths)
            
            # Load current image
            print(f"\n[{idx+1}/{len(all_images)}] {site} - {img_info['season']} {img_info['year']}")
            print(f"Loading: {img_name}")
            img = preloader.get(img_path)
            
            if img is None:
                print(f"ERROR: Could not load {img_path}")
                current_idx += 1
                continue
            
            # Create display
            display_img = create_display_image(img, idx, len(all_images), img_info)
            cv2.imshow(window_name, display_img)
            
            # Wait for user input
            while True:
                key = cv2.waitKey(0) & 0xFF
                
                if key == ord('q'):
                    print("\nQuitting...")
                    raise KeyboardInterrupt
                elif key == ord('x'):
                    label = "no_clouds"
                    break
                elif key == ord('c'):
                    label = "cloudy"
                    break
                elif key == ord('v'):
                    label = "very_cloudy"
                    break
                elif key == ord('s'):
                    print("Skipped")
                    label = None
                    break
                elif key == ord('b'):
                    # Go back to previous image
                    if labeled_history:
                        prev_idx, prev_info, prev_label = labeled_history.pop()
                        prev_site = prev_info['site']
                        prev_name = prev_info['filename']
                        
                        if prev_site in results_by_site and prev_name in results_by_site[prev_site]:
                            del results_by_site[prev_site][prev_name]
                        
                        print(f"Going back to image {prev_idx + 1}: {prev_name}")
                        current_idx = prev_idx
                        label = 'BACK'
                        break
                    else:
                        print("Cannot go back - at first image")
                        continue
                else:
                    print(f"Invalid key. Press X=no clouds, C=cloudy, V=very cloudy, S=skip, B=back, Q=quit")
                    continue
            
            # Handle back command
            if label == 'BACK':
                continue
            
            # Save label
            if label is not None and site in csv_writers:
                _, writer = csv_writers[site]
                writer.writerow([site, img_info['season'], img_info['year'], img_name, label])
                csv_writers[site][0].flush()
                results_by_site[site][img_name] = label
                labeled_history.append((idx, img_info, label))
                print(f"Labeled as: {label}")
            
            current_idx += 1
    
    except KeyboardInterrupt:
        print("\nInterrupted by user")
    finally:
        preloader.cleanup()
        
        # Close all CSV files
        for csvfile, _ in csv_writers.values():
            csvfile.close()
        
        cv2.destroyAllWindows()
        
        # Print summary
        print("\n" + "=" * 60)
        print("LABELING SUMMARY")
        print("=" * 60)
        total_labeled = 0
        for site, results in results_by_site.items():
            count = len(results)
            total_labeled += count
            print(f"{site}: {count} images labeled")
        print(f"\nTotal labeled: {total_labeled} / {len(all_images)}")
        print("=" * 60)


if __name__ == "__main__":
    # Set random seed for reproducibility (optional)
    random.seed(42)
    main()

   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   --------- ------------------------------ 9.2/40.2 MB 57.1 MB/s eta 0:00:01
   -------------------------- ------------- 26.7/40.2 MB 70.6 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 75.2 MB/s  0:00:00
PLANET IMAGE CLOUD LABELER - Random Sample Mode

Sampling 100 images per season per site...
Study sites: YKD_FullCloud, NS_FullCloud, YF_FullCloud

YKD_FullCloud - Breakup: Sampled 88 from 88 images
YKD_FullCloud - Freezeup: Sampled 93 from 93 images
NS_FullCloud - Breakup: Sampled 100 from 100 images
NS_FullCloud - Freezeup: Sampled 100 from 100 images
YF_FullCloud - Breakup: Sampled 99 from 99 images
YF_FullCloud - Freezeup: Sampled 98 from 98 images

Total images to label: 578

[1/578] YKD_FullCloud - Breakup 2024
Loading: 20240506_211829_35_24a7_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[2/578] YKD_FullCloud - Breakup 2020
Loading: 20200422_214622_1011_ortho_analytic_4b_sr.tif

In [ ]:
Collecting opencv-python
  Downloading opencv_python-4.13.0.90-cp37-abi3-win_amd64.whl.metadata (20 kB)
Requirement already satisfied: numpy>=2 in c:\users\nj142\appdata\local\anaconda3\envs\planet-gis\lib\site-packages (from opencv-python) (2.4.1)
Downloading opencv_python-4.13.0.90-cp37-abi3-win_amd64.whl (40.2 MB)
   ---------------------------------------- 0.0/40.2 MB ? eta -:--:--
   --------- ------------------------------ 9.2/40.2 MB 57.1 MB/s eta 0:00:01
   -------------------------- ------------- 26.7/40.2 MB 70.6 MB/s eta 0:00:01
   ---------------------------------------- 40.2/40.2 MB 75.2 MB/s  0:00:00
Installing collected packages: opencv-python
Successfully installed opencv-python-4.13.0.90
============================================================
PLANET IMAGE CLOUD LABELER - Random Sample Mode
============================================================

Sampling 100 images per season per site...
Study sites: YKD_FullCloud, NS_FullCloud, YF_FullCloud

YKD_FullCloud - Breakup: Sampled 88 from 88 images
YKD_FullCloud - Freezeup: Sampled 93 from 93 images
NS_FullCloud - Breakup: Sampled 100 from 100 images
NS_FullCloud - Freezeup: Sampled 100 from 100 images
YF_FullCloud - Breakup: Sampled 99 from 99 images
YF_FullCloud - Freezeup: Sampled 98 from 98 images

Total images to label: 578
============================================================

[1/578] YKD_FullCloud - Breakup 2024
Loading: 20240506_211829_35_24a7_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[2/578] YKD_FullCloud - Breakup 2020
Loading: 20200422_214622_1011_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[3/578] YKD_FullCloud - Breakup 2019
Loading: 20190512_194954_100d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[4/578] YKD_FullCloud - Breakup 2020
Loading: 20200618_214431_0f17_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[5/578] YKD_FullCloud - Breakup 2020
Loading: 20200613_205153_55_1069_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[6/578] YKD_FullCloud - Breakup 2020
Loading: 20200609_184559_100d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[7/578] YKD_FullCloud - Breakup 2020
Loading: 20200430_214442_1011_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[8/578] YKD_FullCloud - Breakup 2019
Loading: 20190624_214117_1025_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[9/578] YKD_FullCloud - Breakup 2023
Loading: 20230423_210923_72_2465_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[10/578] YKD_FullCloud - Breakup 2019
Loading: 20190612_214539_100a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[11/578] YKD_FullCloud - Breakup 2023
Loading: 20230525_210256_65_2447_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[12/578] YKD_FullCloud - Breakup 2021
Loading: 20210618_220610_77_2419_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[13/578] YKD_FullCloud - Breakup 2019
Loading: 20190513_213937_1008_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[14/578] YKD_FullCloud - Breakup 2025
Loading: 20250501_222611_86_24aa_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[15/578] YKD_FullCloud - Breakup 2023
Loading: 20230621_211142_50_2431_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[16/578] YKD_FullCloud - Breakup 2020
Loading: 20200609_184223_0f49_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[17/578] YKD_FullCloud - Breakup 2020
Loading: 20200611_184307_1050_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[18/578] YKD_FullCloud - Breakup 2022
Loading: 20220607_210825_29_2420_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[19/578] YKD_FullCloud - Breakup 2023
Loading: 20230508_210601_18_2442_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[20/578] YKD_FullCloud - Breakup 2020
Loading: 20200528_185002_0f3c_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[21/578] YKD_FullCloud - Breakup 2021
Loading: 20210601_214701_0f4e_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[22/578] YKD_FullCloud - Breakup 2024
Loading: 20240609_212901_26_24bb_ortho_analytic_4b_sr.tif
Going back to image 21: 20210601_214701_0f4e_ortho_analytic_4b_sr.tif

[21/578] YKD_FullCloud - Breakup 2021
Loading: 20210601_214701_0f4e_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[22/578] YKD_FullCloud - Breakup 2024
Loading: 20240609_212901_26_24bb_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[23/578] YKD_FullCloud - Breakup 2022
Loading: 20220415_215316_60_2426_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[24/578] YKD_FullCloud - Breakup 2025
Loading: 20250411_222350_36_2506_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[25/578] YKD_FullCloud - Breakup 2019
Loading: 20190405_194945_101c_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[26/578] YKD_FullCloud - Breakup 2021
Loading: 20210525_214330_1012_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[27/578] YKD_FullCloud - Breakup 2021
Loading: 20210531_222716_69_1064_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[28/578] YKD_FullCloud - Breakup 2019
Loading: 20190529_193955_0f33_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[29/578] YKD_FullCloud - Breakup 2021
Loading: 20210519_212914_59_2264_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[30/578] YKD_FullCloud - Breakup 2023
Loading: 20230430_214309_28_248e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[31/578] YKD_FullCloud - Breakup 2020
Loading: 20200513_214339_101b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[32/578] YKD_FullCloud - Breakup 2025
Loading: 20250627_223330_10_253a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[33/578] YKD_FullCloud - Breakup 2019
Loading: 20190527_214657_1042_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[34/578] YKD_FullCloud - Breakup 2024
Loading: 20240407_211810_19_24bc_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[35/578] YKD_FullCloud - Breakup 2022
Loading: 20220529_214233_52_247b_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[36/578] YKD_FullCloud - Breakup 2022
Loading: 20220610_214229_13_2498_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[37/578] YKD_FullCloud - Breakup 2019
Loading: 20190515_213914_1018_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[38/578] YKD_FullCloud - Breakup 2019
Loading: 20190514_214534_1005_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[39/578] YKD_FullCloud - Breakup 2020
Loading: 20200525_184602_0f49_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[40/578] YKD_FullCloud - Breakup 2022
Loading: 20220521_214744_63_2495_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[41/578] YKD_FullCloud - Breakup 2020
Loading: 20200516_214332_103d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[42/578] YKD_FullCloud - Breakup 2021
Loading: 20210525_211422_66_2456_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[43/578] YKD_FullCloud - Breakup 2021
Loading: 20210401_175424_1052_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[44/578] YKD_FullCloud - Breakup 2020
Loading: 20200429_185244_1048_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[45/578] YKD_FullCloud - Breakup 2019
Loading: 20190505_214532_1021_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[46/578] YKD_FullCloud - Breakup 2023
Loading: 20230425_215014_92_2446_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[47/578] YKD_FullCloud - Breakup 2020
Loading: 20200616_214739_100a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[48/578] YKD_FullCloud - Breakup 2019
Loading: 20190518_214318_1025_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[49/578] YKD_FullCloud - Breakup 2021
Loading: 20210526_202134_99_1065_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[50/578] YKD_FullCloud - Breakup 2021
Loading: 20210526_214431_101b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[51/578] YKD_FullCloud - Breakup 2023
Loading: 20230425_211450_60_24b6_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[52/578] YKD_FullCloud - Breakup 2020
Loading: 20200508_184523_0f36_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[53/578] YKD_FullCloud - Breakup 2020
Loading: 20200517_184337_0f2a_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[54/578] YKD_FullCloud - Breakup 2019
Loading: 20190614_213719_0e16_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[55/578] YKD_FullCloud - Breakup 2023
Loading: 20230619_214947_82_2488_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[56/578] YKD_FullCloud - Breakup 2021
Loading: 20210516_174809_104b_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[57/578] YKD_FullCloud - Breakup 2025
Loading: 20250509_223854_40_2541_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[58/578] YKD_FullCloud - Breakup 2021
Loading: 20210427_211252_74_2450_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[59/578] YKD_FullCloud - Breakup 2021
Loading: 20210629_173429_0f32_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[60/578] YKD_FullCloud - Breakup 2020
Loading: 20200613_214408_1013_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[61/578] YKD_FullCloud - Breakup 2022
Loading: 20220419_224515_73_1058_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[62/578] YKD_FullCloud - Breakup 2021
Loading: 20210513_174636_0f3c_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[63/578] YKD_FullCloud - Breakup 2024
Loading: 20240402_212648_77_2430_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[64/578] YKD_FullCloud - Breakup 2021
Loading: 20210514_203316_33_106e_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[65/578] YKD_FullCloud - Breakup 2019
Loading: 20190527_214311_1027_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[66/578] YKD_FullCloud - Breakup 2024
Loading: 20240623_222400_11_2473_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[67/578] YKD_FullCloud - Breakup 2020
Loading: 20200513_184818_0f32_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[68/578] YKD_FullCloud - Breakup 2023
Loading: 20230502_210612_09_2430_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[69/578] YKD_FullCloud - Breakup 2021
Loading: 20210524_211428_95_241b_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[70/578] YKD_FullCloud - Breakup 2021
Loading: 20210601_213911_1025_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[71/578] YKD_FullCloud - Breakup 2020
Loading: 20200626_214210_0f17_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[72/578] YKD_FullCloud - Breakup 2022
Loading: 20220619_214134_72_2495_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[73/578] YKD_FullCloud - Breakup 2020
Loading: 20200629_214318_0f28_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[74/578] YKD_FullCloud - Breakup 2022
Loading: 20220409_211329_14_2212_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[75/578] YKD_FullCloud - Breakup 2022
Loading: 20220513_220159_88_240c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[76/578] YKD_FullCloud - Breakup 2023
Loading: 20230606_210828_10_24cc_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[77/578] YKD_FullCloud - Breakup 2019
Loading: 20190427_214346_101f_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[78/578] YKD_FullCloud - Breakup 2021
Loading: 20210515_213929_101b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[79/578] YKD_FullCloud - Breakup 2022
Loading: 20220614_211051_33_2457_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[80/578] YKD_FullCloud - Breakup 2020
Loading: 20200424_204728_52_1065_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[81/578] YKD_FullCloud - Breakup 2023
Loading: 20230415_214109_63_249c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[82/578] YKD_FullCloud - Breakup 2021
Loading: 20210525_174243_1048_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[83/578] YKD_FullCloud - Breakup 2020
Loading: 20200529_184535_1049_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[84/578] YKD_FullCloud - Breakup 2020
Loading: 20200508_214248_1011_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[85/578] YKD_FullCloud - Breakup 2020
Loading: 20200613_184146_1049_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[86/578] YKD_FullCloud - Breakup 2020
Loading: 20200616_184149_1_1050_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[87/578] YKD_FullCloud - Breakup 2022
Loading: 20220604_214115_74_2481_ortho_analytic_4b_sr.tif
Going back to image 86: 20200616_184149_1_1050_ortho_analytic_4b_sr.tif

[86/578] YKD_FullCloud - Breakup 2020
Loading: 20200616_184149_1_1050_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[87/578] YKD_FullCloud - Breakup 2022
Loading: 20220604_214115_74_2481_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[88/578] YKD_FullCloud - Breakup 2022
Loading: 20220430_210508_65_2455_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[89/578] YKD_FullCloud - Freezeup 2019
Loading: 20191013_215740_60_1066_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[90/578] YKD_FullCloud - Freezeup 2020
Loading: 20201031_214624_1039_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[91/578] YKD_FullCloud - Freezeup 2019
Loading: 20191005_214418_1040_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[92/578] YKD_FullCloud - Freezeup 2021
Loading: 20211031_215233_1026_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[93/578] YKD_FullCloud - Freezeup 2022
Loading: 20221008_211149_35_241d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[94/578] YKD_FullCloud - Freezeup 2021
Loading: 20211011_213727_103c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[95/578] YKD_FullCloud - Freezeup 2019
Loading: 20191016_192403_1052_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[96/578] YKD_FullCloud - Freezeup 2020
Loading: 20201030_204133_81_106e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[97/578] YKD_FullCloud - Freezeup 2023
Loading: 20231022_211246_83_24bf_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[98/578] YKD_FullCloud - Freezeup 2025
Loading: 20251103_223027_59_24b8_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[99/578] YKD_FullCloud - Freezeup 2025
Loading: 20251019_224430_14_24f2_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[100/578] YKD_FullCloud - Freezeup 2022
Loading: 20221028_213829_77_249d_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[101/578] YKD_FullCloud - Freezeup 2022
Loading: 20221002_215818_20_2414_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[102/578] YKD_FullCloud - Freezeup 2022
Loading: 20221019_215251_53_241c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[103/578] YKD_FullCloud - Freezeup 2020
Loading: 20201012_221336_02_240f_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[104/578] YKD_FullCloud - Freezeup 2021
Loading: 20211001_212026_01_2453_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[105/578] YKD_FullCloud - Freezeup 2020
Loading: 20201012_214709_1011_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[106/578] YKD_FullCloud - Freezeup 2020
Loading: 20201101_215000_0f28_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[107/578] YKD_FullCloud - Freezeup 2023
Loading: 20231021_211644_74_24a8_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[108/578] YKD_FullCloud - Freezeup 2022
Loading: 20221111_214056_49_2478_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[109/578] YKD_FullCloud - Freezeup 2024
Loading: 20241025_213137_73_24ce_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[110/578] YKD_FullCloud - Freezeup 2022
Loading: 20221015_215637_39_2416_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[111/578] YKD_FullCloud - Freezeup 2025
Loading: 20251030_225301_63_24d3_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[112/578] YKD_FullCloud - Freezeup 2021
Loading: 20211110_213436_1011_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[113/578] YKD_FullCloud - Freezeup 2020
Loading: 20201031_214622_1039_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[114/578] YKD_FullCloud - Freezeup 2023
Loading: 20231110_211511_20_242e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[115/578] YKD_FullCloud - Freezeup 2022
Loading: 20221030_214540_47_247e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[116/578] YKD_FullCloud - Freezeup 2024
Loading: 20241111_221725_52_24cd_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[117/578] YKD_FullCloud - Freezeup 2019
Loading: 20191024_220244_02_105c_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[118/578] YKD_FullCloud - Freezeup 2019
Loading: 20191010_214535_0f17_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[119/578] YKD_FullCloud - Freezeup 2022
Loading: 20221018_210200_14_2460_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[120/578] YKD_FullCloud - Freezeup 2025
Loading: 20251112_223653_66_2506_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[121/578] YKD_FullCloud - Freezeup 2019
Loading: 20191016_213900_1001_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[122/578] YKD_FullCloud - Freezeup 2024
Loading: 20241115_222432_96_24da_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[123/578] YKD_FullCloud - Freezeup 2019
Loading: 20191021_220001_28_1061_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[124/578] YKD_FullCloud - Freezeup 2024
Loading: 20241107_221934_80_24ee_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[125/578] YKD_FullCloud - Freezeup 2021
Loading: 20211105_214140_1003_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[126/578] YKD_FullCloud - Freezeup 2024
Loading: 20241111_221746_93_24e1_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[127/578] YKD_FullCloud - Freezeup 2021
Loading: 20211025_220343_09_240c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[128/578] YKD_FullCloud - Freezeup 2025
Loading: 20251108_223529_63_250e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[129/578] YKD_FullCloud - Freezeup 2020
Loading: 20201027_214920_0f4e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[130/578] YKD_FullCloud - Freezeup 2022
Loading: 20221013_215521_14_2254_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[131/578] YKD_FullCloud - Freezeup 2023
Loading: 20231031_215429_70_2475_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[132/578] YKD_FullCloud - Freezeup 2025
Loading: 20251112_223006_10_2549_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[133/578] YKD_FullCloud - Freezeup 2025
Loading: 20251018_223559_96_2540_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[134/578] YKD_FullCloud - Freezeup 2020
Loading: 20201009_214127_0f28_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[135/578] YKD_FullCloud - Freezeup 2021
Loading: 20211015_220628_92_2424_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[136/578] YKD_FullCloud - Freezeup 2019
Loading: 20191001_214507_0f25_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[137/578] YKD_FullCloud - Freezeup 2022
Loading: 20221018_213556_61_2461_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[138/578] YKD_FullCloud - Freezeup 2022
Loading: 20221021_214439_42_24a3_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[139/578] YKD_FullCloud - Freezeup 2025
Loading: 20251024_224538_75_24f2_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[140/578] YKD_FullCloud - Freezeup 2022
Loading: 20221103_213820_67_249d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[141/578] YKD_FullCloud - Freezeup 2020
Loading: 20201021_214024_101b_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[142/578] YKD_FullCloud - Freezeup 2021
Loading: 20211106_214543_1011_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[143/578] YKD_FullCloud - Freezeup 2024
Loading: 20241029_213258_46_2417_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[144/578] YKD_FullCloud - Freezeup 2022
Loading: 20221022_214657_68_2479_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[145/578] YKD_FullCloud - Freezeup 2024
Loading: 20241104_221807_03_2516_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[146/578] YKD_FullCloud - Freezeup 2021
Loading: 20211115_223456_72_1064_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[147/578] YKD_FullCloud - Freezeup 2021
Loading: 20211107_213110_72_2276_ortho_analytic_4b_sr.tif
Going back to image 146: 20211115_223456_72_1064_ortho_analytic_4b_sr.tif

[146/578] YKD_FullCloud - Freezeup 2021
Loading: 20211115_223456_72_1064_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[147/578] YKD_FullCloud - Freezeup 2021
Loading: 20211107_213110_72_2276_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[148/578] YKD_FullCloud - Freezeup 2021
Loading: 20211112_220036_09_2403_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[149/578] YKD_FullCloud - Freezeup 2020
Loading: 20201102_222131_70_1064_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[150/578] YKD_FullCloud - Freezeup 2022
Loading: 20221030_214118_21_248c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[151/578] YKD_FullCloud - Freezeup 2021
Loading: 20211112_220154_74_2274_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[152/578] YKD_FullCloud - Freezeup 2023
Loading: 20231019_211028_10_24b5_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[153/578] YKD_FullCloud - Freezeup 2019
Loading: 20191004_214927_100a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[154/578] YKD_FullCloud - Freezeup 2021
Loading: 20211024_220121_92_2426_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[155/578] YKD_FullCloud - Freezeup 2020
Loading: 20201020_214928_1040_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[156/578] YKD_FullCloud - Freezeup 2022
Loading: 20221020_210353_66_241f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[157/578] YKD_FullCloud - Freezeup 2020
Loading: 20201028_222500_83_105d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[158/578] YKD_FullCloud - Freezeup 2020
Loading: 20201101_204050_85_1069_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[159/578] YKD_FullCloud - Freezeup 2020
Loading: 20201020_204303_07_1069_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[160/578] YKD_FullCloud - Freezeup 2022
Loading: 20221031_211003_80_241f_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[161/578] YKD_FullCloud - Freezeup 2022
Loading: 20221014_214020_71_2486_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[162/578] YKD_FullCloud - Freezeup 2023
Loading: 20231031_215431_75_2475_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[163/578] YKD_FullCloud - Freezeup 2019
Loading: 20191009_213932_1034_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[164/578] YKD_FullCloud - Freezeup 2021
Loading: 20211101_223931_70_105a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[165/578] YKD_FullCloud - Freezeup 2020
Loading: 20201022_204300_53_106e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[166/578] YKD_FullCloud - Freezeup 2021
Loading: 20211105_214138_1003_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[167/578] YKD_FullCloud - Freezeup 2020
Loading: 20201028_212515_19_2259_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[168/578] YKD_FullCloud - Freezeup 2022
Loading: 20221019_213846_74_2470_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[169/578] YKD_FullCloud - Freezeup 2021
Loading: 20211031_211255_18_2206_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[170/578] YKD_FullCloud - Freezeup 2019
Loading: 20191106_220206_99_1060_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[171/578] YKD_FullCloud - Freezeup 2019
Loading: 20191002_192643_0f46_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[172/578] YKD_FullCloud - Freezeup 2019
Loading: 20191101_213822_101f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[173/578] YKD_FullCloud - Freezeup 2023
Loading: 20231020_211415_87_2458_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[174/578] YKD_FullCloud - Freezeup 2023
Loading: 20231022_211249_13_24bf_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[175/578] YKD_FullCloud - Freezeup 2020
Loading: 20201021_214339_103c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[176/578] YKD_FullCloud - Freezeup 2021
Loading: 20211024_214128_1014_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[177/578] YKD_FullCloud - Freezeup 2025
Loading: 20251023_223101_53_2507_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[178/578] YKD_FullCloud - Freezeup 2022
Loading: 20221020_210348_97_241f_ortho_analytic_4b_sr.tif
Going back to image 177: 20251023_223101_53_2507_ortho_analytic_4b_sr.tif

[177/578] YKD_FullCloud - Freezeup 2025
Loading: 20251023_223101_53_2507_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[178/578] YKD_FullCloud - Freezeup 2022
Loading: 20221020_210348_97_241f_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[179/578] YKD_FullCloud - Freezeup 2019
Loading: 20191027_210229_70_1068_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[180/578] YKD_FullCloud - Freezeup 2019
Loading: 20191104_220136_29_105e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[181/578] YKD_FullCloud - Freezeup 2019
Loading: 20191004_192307_0f36_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[182/578] NS_FullCloud - Breakup 2025
Loading: 20250627_222710_77_24fa_ortho_analytic_4b_sr.tif
Going back to image 181: 20191004_192307_0f36_ortho_analytic_4b_sr.tif

[181/578] YKD_FullCloud - Freezeup 2019
Loading: 20191004_192307_0f36_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[182/578] NS_FullCloud - Breakup 2025
Loading: 20250627_222710_77_24fa_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[183/578] NS_FullCloud - Breakup 2023
Loading: 20230526_215444_53_248e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[184/578] NS_FullCloud - Breakup 2020
Loading: 20200521_174544_104a_ortho_analytic_3b.tif
Labeled as: very_cloudy

[185/578] NS_FullCloud - Breakup 2025
Loading: 20250713_223435_13_2516_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[186/578] NS_FullCloud - Breakup 2024
Loading: 20240529_212542_95_24c4_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[187/578] NS_FullCloud - Breakup 2022
Loading: 20220527_215114_31_249d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[188/578] NS_FullCloud - Breakup 2023
Loading: 20230602_215453_91_2470_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[189/578] NS_FullCloud - Breakup 2020
Loading: 20200602_175107_1049_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[190/578] NS_FullCloud - Breakup 2020
Loading: 20200710_194205_17_1067_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[191/578] NS_FullCloud - Breakup 2023
Loading: 20230522_211501_67_2415_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[192/578] NS_FullCloud - Breakup 2023
Loading: 20230701_215910_27_2470_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[193/578] NS_FullCloud - Breakup 2021
Loading: 20210725_214022_0f15_ortho_analytic_3b.tif
Labeled as: very_cloudy

[194/578] NS_FullCloud - Breakup 2020
Loading: 20200617_194650_70_1063_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[195/578] NS_FullCloud - Breakup 2023
Loading: 20230530_215435_13_248b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[196/578] NS_FullCloud - Breakup 2020
Loading: 20200613_174413_0f2b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[197/578] NS_FullCloud - Breakup 2020
Loading: 20200721_174340_1054_ortho_analytic_3b.tif
Labeled as: very_cloudy

[198/578] NS_FullCloud - Breakup 2021
Loading: 20210716_212009_53_2447_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[199/578] NS_FullCloud - Breakup 2021
Loading: 20210702_214600_101b_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[200/578] NS_FullCloud - Breakup 2022
Loading: 20220522_212459_92_2262_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[201/578] NS_FullCloud - Breakup 2023
Loading: 20230522_211021_30_24c5_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[202/578] NS_FullCloud - Breakup 2022
Loading: 20220523_211554_44_2442_ortho_analytic_4b_sr.tif
Going back to image 201: 20230522_211021_30_24c5_ortho_analytic_4b_sr.tif

[201/578] NS_FullCloud - Breakup 2023
Loading: 20230522_211021_30_24c5_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[202/578] NS_FullCloud - Breakup 2022
Loading: 20220523_211554_44_2442_ortho_analytic_4b_sr.tif
Going back to image 201: 20230522_211021_30_24c5_ortho_analytic_4b_sr.tif

[201/578] NS_FullCloud - Breakup 2023
Loading: 20230522_211021_30_24c5_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[202/578] NS_FullCloud - Breakup 2022
Loading: 20220523_211554_44_2442_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[203/578] NS_FullCloud - Breakup 2020
Loading: 20200520_195505_35_106d_ortho_analytic_4b_sr.tif
Going back to image 202: 20220523_211554_44_2442_ortho_analytic_4b_sr.tif

[202/578] NS_FullCloud - Breakup 2022
Loading: 20220523_211554_44_2442_ortho_analytic_4b_sr.tif
Going back to image 201: 20230522_211021_30_24c5_ortho_analytic_4b_sr.tif

[201/578] NS_FullCloud - Breakup 2023
Loading: 20230522_211021_30_24c5_ortho_analytic_4b_sr.tif
Going back to image 200: 20220522_212459_92_2262_ortho_analytic_4b_sr.tif

[200/578] NS_FullCloud - Breakup 2022
Loading: 20220522_212459_92_2262_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[201/578] NS_FullCloud - Breakup 2023
Loading: 20230522_211021_30_24c5_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[202/578] NS_FullCloud - Breakup 2022
Loading: 20220523_211554_44_2442_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[203/578] NS_FullCloud - Breakup 2020
Loading: 20200520_195505_35_106d_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[204/578] NS_FullCloud - Breakup 2020
Loading: 20200703_065419_103c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[205/578] NS_FullCloud - Breakup 2020
Loading: 20200625_215017_101b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[206/578] NS_FullCloud - Breakup 2019
Loading: 20190619_215420_1001_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[207/578] NS_FullCloud - Breakup 2021
Loading: 20210530_212908_47_2262_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[208/578] NS_FullCloud - Breakup 2019
Loading: 20190524_214137_103c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[209/578] NS_FullCloud - Breakup 2024
Loading: 20240623_065100_26_2429_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[210/578] NS_FullCloud - Breakup 2020
Loading: 20200629_174612_0f46_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[211/578] NS_FullCloud - Breakup 2023
Loading: 20230626_212216_15_2427_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[212/578] NS_FullCloud - Breakup 2019
Loading: 20190515_185030_0f2e_ortho_analytic_3b.tif
Labeled as: very_cloudy

[213/578] NS_FullCloud - Breakup 2019
Loading: 20190627_214701_1009_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[214/578] NS_FullCloud - Breakup 2019
Loading: 20190607_214729_1001_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[215/578] NS_FullCloud - Breakup 2023
Loading: 20230602_215633_51_248c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[216/578] NS_FullCloud - Breakup 2023
Loading: 20230621_215538_88_248b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[217/578] NS_FullCloud - Breakup 2019
Loading: 20190529_214719_1032_ortho_analytic_3b.tif
Labeled as: very_cloudy

[218/578] NS_FullCloud - Breakup 2021
Loading: 20210523_213014_32_227e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[219/578] NS_FullCloud - Breakup 2022
Loading: 20220706_212216_24_241f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[220/578] NS_FullCloud - Breakup 2020
Loading: 20200705_174226_1050_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[221/578] NS_FullCloud - Breakup 2023
Loading: 20230706_211703_31_24ca_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[222/578] NS_FullCloud - Breakup 2020
Loading: 20200521_175326_1050_ortho_analytic_3b.tif
Labeled as: very_cloudy

[223/578] NS_FullCloud - Breakup 2022
Loading: 20220617_061634_17_2460_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[224/578] NS_FullCloud - Breakup 2024
Loading: 20240618_062855_66_24c9_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[225/578] NS_FullCloud - Breakup 2020
Loading: 20200518_212822_72_2257_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[226/578] NS_FullCloud - Breakup 2020
Loading: 20200710_212116_39_2263_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[227/578] NS_FullCloud - Breakup 2023
Loading: 20230520_210312_90_2445_ortho_analytic_4b_sr.tif
Going back to image 226: 20200710_212116_39_2263_ortho_analytic_4b_sr.tif

[226/578] NS_FullCloud - Breakup 2020
Loading: 20200710_212116_39_2263_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[227/578] NS_FullCloud - Breakup 2023
Loading: 20230520_210312_90_2445_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[228/578] NS_FullCloud - Breakup 2021
Loading: 20210609_214140_1011_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[229/578] NS_FullCloud - Breakup 2020
Loading: 20200718_173733_100d_ortho_analytic_3b.tif
Labeled as: very_cloudy

[230/578] NS_FullCloud - Breakup 2021
Loading: 20210719_215032_1039_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[231/578] NS_FullCloud - Breakup 2020
Loading: 20200702_194025_31_1067_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[232/578] NS_FullCloud - Breakup 2024
Loading: 20240627_222359_55_24c6_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[233/578] NS_FullCloud - Breakup 2021
Loading: 20210713_164027_0f21_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[234/578] NS_FullCloud - Breakup 2020
Loading: 20200613_174419_0f2b_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[235/578] NS_FullCloud - Breakup 2020
Loading: 20200516_175402_1050_ortho_analytic_3b.tif
Labeled as: very_cloudy

[236/578] NS_FullCloud - Breakup 2019
Loading: 20190604_184118_0f49_ortho_analytic_3b.tif
Labeled as: very_cloudy

[237/578] NS_FullCloud - Breakup 2021
Loading: 20210607_215518_75_105c_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[238/578] NS_FullCloud - Breakup 2022
Loading: 20220523_214923_82_247d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[239/578] NS_FullCloud - Breakup 2024
Loading: 20240613_221525_66_2490_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[240/578] NS_FullCloud - Breakup 2020
Loading: 20200602_215611_1032_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[241/578] NS_FullCloud - Breakup 2023
Loading: 20230617_220222_68_2424_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[242/578] NS_FullCloud - Breakup 2024
Loading: 20240515_213741_44_2459_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[243/578] NS_FullCloud - Breakup 2023
Loading: 20230716_212154_75_24a1_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[244/578] NS_FullCloud - Breakup 2019
Loading: 20190528_184423_0f33_ortho_analytic_3b.tif
Labeled as: very_cloudy

[245/578] NS_FullCloud - Breakup 2021
Loading: 20210607_215516_87_105c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[246/578] NS_FullCloud - Breakup 2020
Loading: 20200718_215200_1013_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[247/578] NS_FullCloud - Breakup 2024
Loading: 20240611_221218_82_227a_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[248/578] NS_FullCloud - Breakup 2024
Loading: 20240621_213628_71_2430_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[249/578] NS_FullCloud - Breakup 2024
Loading: 20240523_214426_46_2439_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[250/578] NS_FullCloud - Breakup 2021
Loading: 20210712_215854_1026_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[251/578] NS_FullCloud - Breakup 2022
Loading: 20220530_211234_03_2442_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[252/578] NS_FullCloud - Breakup 2021
Loading: 20210712_215059_100c_ortho_analytic_4b_sr.tif
Going back to image 251: 20220530_211234_03_2442_ortho_analytic_4b_sr.tif

[251/578] NS_FullCloud - Breakup 2022
Loading: 20220530_211234_03_2442_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[252/578] NS_FullCloud - Breakup 2021
Loading: 20210712_215059_100c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[253/578] NS_FullCloud - Breakup 2022
Loading: 20220526_212146_07_2434_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[254/578] NS_FullCloud - Breakup 2020
Loading: 20200519_195228_03_1062_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[255/578] NS_FullCloud - Breakup 2022
Loading: 20220530_214144_33_248c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[256/578] NS_FullCloud - Breakup 2023
Loading: 20230719_211710_40_242b_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[257/578] NS_FullCloud - Breakup 2019
Loading: 20190529_214723_1032_ortho_analytic_3b.tif
Labeled as: very_cloudy

[258/578] NS_FullCloud - Breakup 2024
Loading: 20240617_062915_71_24b0_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[259/578] NS_FullCloud - Breakup 2020
Loading: 20200727_173955_0f24_ortho_analytic_3b.tif
Labeled as: very_cloudy

[260/578] NS_FullCloud - Breakup 2024
Loading: 20240618_213903_41_242d_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[261/578] NS_FullCloud - Breakup 2023
Loading: 20230610_061543_72_242b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[262/578] NS_FullCloud - Breakup 2021
Loading: 20210523_164645_1_100d_ortho_analytic_3b.tif
Labeled as: very_cloudy

[263/578] NS_FullCloud - Breakup 2023
Loading: 20230607_215440_20_248f_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[264/578] NS_FullCloud - Breakup 2020
Loading: 20200717_174344_104b_ortho_analytic_3b.tif
Labeled as: very_cloudy

[265/578] NS_FullCloud - Breakup 2019
Loading: 20190519_214125_1034_ortho_analytic_3b.tif
Labeled as: very_cloudy

[266/578] NS_FullCloud - Breakup 2024
Loading: 20240613_063720_68_24c8_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[267/578] NS_FullCloud - Breakup 2025
Loading: 20250530_222732_05_2516_ortho_analytic_4b_sr.tif
Going back to image 266: 20240613_063720_68_24c8_ortho_analytic_4b_sr.tif

[266/578] NS_FullCloud - Breakup 2024
Loading: 20240613_063720_68_24c8_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[267/578] NS_FullCloud - Breakup 2025
Loading: 20250530_222732_05_2516_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[268/578] NS_FullCloud - Breakup 2021
Loading: 20210723_214239_1039_ortho_analytic_3b.tif
Labeled as: very_cloudy

[269/578] NS_FullCloud - Breakup 2020
Loading: 20200523_174911_1020_ortho_analytic_3b.tif
Labeled as: very_cloudy

[270/578] NS_FullCloud - Breakup 2020
Loading: 20200602_174706_1_1020_ortho_analytic_3b.tif
Labeled as: very_cloudy

[271/578] NS_FullCloud - Breakup 2020
Loading: 20200719_174132_0f4d_ortho_analytic_3b.tif
Labeled as: very_cloudy

[272/578] NS_FullCloud - Breakup 2024
Loading: 20240618_213907_05_242d_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[273/578] NS_FullCloud - Breakup 2025
Loading: 20250713_223430_69_2516_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[274/578] NS_FullCloud - Breakup 2022
Loading: 20220515_221303_33_2405_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[275/578] NS_FullCloud - Breakup 2019
Loading: 20190709_214612_1032_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[276/578] NS_FullCloud - Breakup 2020
Loading: 20200605_214533_1004_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[277/578] NS_FullCloud - Breakup 2023
Loading: 20230716_212115_11_24ba_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[278/578] NS_FullCloud - Breakup 2020
Loading: 20200529_175135_0f49_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[279/578] NS_FullCloud - Breakup 2020
Loading: 20200612_220051_1105_ortho_analytic_3b.tif
Labeled as: cloudy

[280/578] NS_FullCloud - Breakup 2024
Loading: 20240701_222234_70_24d2_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[281/578] NS_FullCloud - Breakup 2019
Loading: 20190724_213757_0e2f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[282/578] NS_FullCloud - Freezeup 2021
Loading: 20211002_220541_24_240f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[283/578] NS_FullCloud - Freezeup 2022
Loading: 20221007_214203_98_248b_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[284/578] NS_FullCloud - Freezeup 2021
Loading: 20211004_213453_03_227e_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[285/578] NS_FullCloud - Freezeup 2022
Loading: 20221004_211418_50_2420_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[286/578] NS_FullCloud - Freezeup 2024
Loading: 20241007_221638_08_24aa_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[287/578] NS_FullCloud - Freezeup 2024
Loading: 20241015_213111_36_24a8_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[288/578] NS_FullCloud - Freezeup 2023
Loading: 20231003_211257_43_2465_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[289/578] NS_FullCloud - Freezeup 2023
Loading: 20231013_213155_24_24c1_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[290/578] NS_FullCloud - Freezeup 2024
Loading: 20241011_223144_82_2409_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[291/578] NS_FullCloud - Freezeup 2022
Loading: 20221010_214929_50_2477_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[292/578] NS_FullCloud - Freezeup 2020
Loading: 20201005_214539_1035_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[293/578] NS_FullCloud - Freezeup 2020
Loading: 20201008_215455_1040_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[294/578] NS_FullCloud - Freezeup 2021
Loading: 20211004_215003_1038_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[295/578] NS_FullCloud - Freezeup 2020
Loading: 20201010_222401_95_105a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[296/578] NS_FullCloud - Freezeup 2019
Loading: 20191010_215411_1014_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[297/578] NS_FullCloud - Freezeup 2023
Loading: 20231005_211756_69_24b3_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[298/578] NS_FullCloud - Freezeup 2022
Loading: 20221016_214717_37_2495_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[299/578] NS_FullCloud - Freezeup 2023
Loading: 20231014_212137_11_24d0_ortho_analytic_4b_sr.tif
Going back to image 298: 20221016_214717_37_2495_ortho_analytic_4b_sr.tif

[298/578] NS_FullCloud - Freezeup 2022
Loading: 20221016_214717_37_2495_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[299/578] NS_FullCloud - Freezeup 2023
Loading: 20231014_212137_11_24d0_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[300/578] NS_FullCloud - Freezeup 2021
Loading: 20211008_213930_1039_ortho_analytic_3b.tif
Labeled as: very_cloudy

[301/578] NS_FullCloud - Freezeup 2023
Loading: 20231011_215640_62_2483_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[302/578] NS_FullCloud - Freezeup 2019
Loading: 20191010_215410_1014_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[303/578] NS_FullCloud - Freezeup 2024
Loading: 20241011_223147_18_2409_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[304/578] NS_FullCloud - Freezeup 2022
Loading: 20221009_215741_32_2438_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[305/578] NS_FullCloud - Freezeup 2022
Loading: 20221011_220028_01_227b_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[306/578] NS_FullCloud - Freezeup 2022
Loading: 20221014_215550_83_24a3_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[307/578] NS_FullCloud - Freezeup 2020
Loading: 20201005_215828_1008_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[308/578] NS_FullCloud - Freezeup 2023
Loading: 20231010_220708_29_2479_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[309/578] NS_FullCloud - Freezeup 2022
Loading: 20221012_214545_75_247d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[310/578] NS_FullCloud - Freezeup 2019
Loading: 20191012_214915_100a_ortho_analytic_3b.tif
Labeled as: very_cloudy

[311/578] NS_FullCloud - Freezeup 2020
Loading: 20201007_221924_85_105d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[312/578] NS_FullCloud - Freezeup 2019
Loading: 20191010_215416_1014_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[313/578] NS_FullCloud - Freezeup 2023
Loading: 20231013_211823_07_24c2_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[314/578] NS_FullCloud - Freezeup 2020
Loading: 20201013_215458_1014_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[315/578] NS_FullCloud - Freezeup 2022
Loading: 20221002_210846_25_241f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[316/578] NS_FullCloud - Freezeup 2019
Loading: 20191013_220913_18_1061_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[317/578] NS_FullCloud - Freezeup 2020
Loading: 20201013_215506_1014_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[318/578] NS_FullCloud - Freezeup 2019
Loading: 20191010_200557_26_1065_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[319/578] NS_FullCloud - Freezeup 2021
Loading: 20211006_214306_0f15_ortho_analytic_3b.tif
Labeled as: very_cloudy

[320/578] NS_FullCloud - Freezeup 2022
Loading: 20221011_215137_39_2478_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[321/578] NS_FullCloud - Freezeup 2020
Loading: 20201010_215147_1010_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[322/578] NS_FullCloud - Freezeup 2021
Loading: 20211010_223501_74_1061_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[323/578] NS_FullCloud - Freezeup 2023
Loading: 20231015_212703_70_24b0_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[324/578] NS_FullCloud - Freezeup 2025
Loading: 20251009_224940_10_24d1_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[325/578] NS_FullCloud - Freezeup 2025
Loading: 20251014_224651_29_253d_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[326/578] NS_FullCloud - Freezeup 2023
Loading: 20231009_211624_59_2465_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[327/578] NS_FullCloud - Freezeup 2020
Loading: 20201001_214814_1035_ortho_analytic_3b.tif
Labeled as: very_cloudy

[328/578] NS_FullCloud - Freezeup 2019
Loading: 20191012_215617_1011_ortho_analytic_3b.tif
Labeled as: very_cloudy

[329/578] NS_FullCloud - Freezeup 2022
Loading: 20221007_214208_70_248b_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[330/578] NS_FullCloud - Freezeup 2021
Loading: 20211012_214744_1034_ortho_analytic_3b.tif
Labeled as: very_cloudy

[331/578] NS_FullCloud - Freezeup 2022
Loading: 20221004_211427_64_2420_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[332/578] NS_FullCloud - Freezeup 2023
Loading: 20231003_212219_11_2439_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[333/578] NS_FullCloud - Freezeup 2025
Loading: 20251007_223709_56_251a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[334/578] NS_FullCloud - Freezeup 2020
Loading: 20201009_222106_96_1066_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[335/578] NS_FullCloud - Freezeup 2022
Loading: 20221016_214001_63_247c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[336/578] NS_FullCloud - Freezeup 2022
Loading: 20221002_213343_59_2262_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[337/578] NS_FullCloud - Freezeup 2021
Loading: 20211008_221229_1105_ortho_analytic_3b.tif
Labeled as: very_cloudy

[338/578] NS_FullCloud - Freezeup 2025
Loading: 20251007_222937_07_250e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[339/578] NS_FullCloud - Freezeup 2020
Loading: 20201011_222436_92_1058_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[340/578] NS_FullCloud - Freezeup 2022
Loading: 20221001_211213_11_2429_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[341/578] NS_FullCloud - Freezeup 2019
Loading: 20191009_220806_51_105c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[342/578] NS_FullCloud - Freezeup 2019
Loading: 20191001_221236_13_1066_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[343/578] NS_FullCloud - Freezeup 2022
Loading: 20221014_215546_12_24a3_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[344/578] NS_FullCloud - Freezeup 2022
Loading: 20221006_220712_46_2414_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[345/578] NS_FullCloud - Freezeup 2023
Loading: 20231009_215407_64_2477_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[346/578] NS_FullCloud - Freezeup 2023
Loading: 20231010_220749_39_2488_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[347/578] NS_FullCloud - Freezeup 2021
Loading: 20211003_212657_08_242d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[348/578] NS_FullCloud - Freezeup 2022
Loading: 20221002_220623_28_240c_ortho_analytic_4b_sr.tif
Going back to image 347: 20211003_212657_08_242d_ortho_analytic_4b_sr.tif

[347/578] NS_FullCloud - Freezeup 2021
Loading: 20211003_212657_08_242d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[348/578] NS_FullCloud - Freezeup 2022
Loading: 20221002_220623_28_240c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[349/578] NS_FullCloud - Freezeup 2021
Loading: 20211001_215028_1034_ortho_analytic_3b.tif
Labeled as: very_cloudy

[350/578] NS_FullCloud - Freezeup 2021
Loading: 20211013_214910_0f15_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[351/578] NS_FullCloud - Freezeup 2021
Loading: 20211004_211903_67_2440_ortho_analytic_4b_sr.tif
Going back to image 350: 20211013_214910_0f15_ortho_analytic_4b_sr.tif

[350/578] NS_FullCloud - Freezeup 2021
Loading: 20211013_214910_0f15_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[351/578] NS_FullCloud - Freezeup 2021
Loading: 20211004_211903_67_2440_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[352/578] NS_FullCloud - Freezeup 2021
Loading: 20211006_213822_1014_ortho_analytic_3b.tif
Labeled as: very_cloudy

[353/578] NS_FullCloud - Freezeup 2019
Loading: 20191012_214916_100a_ortho_analytic_3b.tif
Labeled as: very_cloudy

[354/578] NS_FullCloud - Freezeup 2019
Loading: 20191004_214631_0f4e_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[355/578] NS_FullCloud - Freezeup 2023
Loading: 20231004_215332_03_2488_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[356/578] NS_FullCloud - Freezeup 2020
Loading: 20201011_214542_1001_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[357/578] NS_FullCloud - Freezeup 2019
Loading: 20191010_215823_1008_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[358/578] NS_FullCloud - Freezeup 2023
Loading: 20231008_220000_72_2498_ortho_analytic_4b_sr.tif
Going back to image 357: 20191010_215823_1008_ortho_analytic_4b_sr.tif

[357/578] NS_FullCloud - Freezeup 2019
Loading: 20191010_215823_1008_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[358/578] NS_FullCloud - Freezeup 2023
Loading: 20231008_220000_72_2498_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[359/578] NS_FullCloud - Freezeup 2019
Loading: 20191013_215525_0f52_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[360/578] NS_FullCloud - Freezeup 2020
Loading: 20201001_214815_1035_ortho_analytic_3b.tif
Labeled as: very_cloudy

[361/578] NS_FullCloud - Freezeup 2024
Loading: 20241001_214045_69_24c8_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[362/578] NS_FullCloud - Freezeup 2021
Loading: 20211010_223503_24_1061_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[363/578] NS_FullCloud - Freezeup 2023
Loading: 20231007_211350_18_24b4_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[364/578] NS_FullCloud - Freezeup 2024
Loading: 20241011_222605_39_251d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[365/578] NS_FullCloud - Freezeup 2021
Loading: 20211010_223506_24_1061_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[366/578] NS_FullCloud - Freezeup 2019
Loading: 20191009_214518_0f25_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[367/578] NS_FullCloud - Freezeup 2020
Loading: 20201007_215219_1025_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[368/578] NS_FullCloud - Freezeup 2023
Loading: 20231015_212701_39_24b0_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[369/578] NS_FullCloud - Freezeup 2023
Loading: 20231006_211459_51_2465_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[370/578] NS_FullCloud - Freezeup 2019
Loading: 20191002_215631_0f22_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[371/578] NS_FullCloud - Freezeup 2021
Loading: 20211013_212328_81_2456_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[372/578] NS_FullCloud - Freezeup 2021
Loading: 20211013_212326_51_2456_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[373/578] NS_FullCloud - Freezeup 2022
Loading: 20221004_215126_59_2483_ortho_analytic_4b_sr.tif
Going back to image 372: 20211013_212326_51_2456_ortho_analytic_4b_sr.tif

[372/578] NS_FullCloud - Freezeup 2021
Loading: 20211013_212326_51_2456_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[373/578] NS_FullCloud - Freezeup 2022
Loading: 20221004_215126_59_2483_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[374/578] NS_FullCloud - Freezeup 2019
Loading: 20191012_215615_1011_ortho_analytic_3b.tif
Labeled as: very_cloudy

[375/578] NS_FullCloud - Freezeup 2020
Loading: 20201002_215630_1010_ortho_analytic_3b.tif
Labeled as: very_cloudy

[376/578] NS_FullCloud - Freezeup 2022
Loading: 20221008_214606_33_2498_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[377/578] NS_FullCloud - Freezeup 2025
Loading: 20251004_224306_81_24fe_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[378/578] NS_FullCloud - Freezeup 2025
Loading: 20251014_224250_03_24e9_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[379/578] NS_FullCloud - Freezeup 2020
Loading: 20201007_194424_25_106c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[380/578] NS_FullCloud - Freezeup 2023
Loading: 20231001_221931_25_2414_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[381/578] NS_FullCloud - Freezeup 2021
Loading: 20211014_215335_103c_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[382/578] YF_FullCloud - Breakup 2020
Loading: 20200604_205117_0f25_ortho_analytic_3b.tif
Labeled as: very_cloudy

[383/578] YF_FullCloud - Breakup 2019
Loading: 20190503_205518_1021_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[384/578] YF_FullCloud - Breakup 2019
Loading: 20190605_205625_101f_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[385/578] YF_FullCloud - Breakup 2024
Loading: 20240501_203803_68_24c4_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[386/578] YF_FullCloud - Breakup 2021
Loading: 20210616_205909_0f15_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[387/578] YF_FullCloud - Breakup 2020
Loading: 20200608_172205_104a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[388/578] YF_FullCloud - Breakup 2019
Loading: 20190503_205326_100c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[389/578] YF_FullCloud - Breakup 2019
Loading: 20190421_205150_1105_ortho_analytic_3b.tif
Labeled as: cloudy

[390/578] YF_FullCloud - Breakup 2020
Loading: 20200703_193245_94_106c_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[391/578] YF_FullCloud - Breakup 2019
Loading: 20190619_205809_0f25_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[392/578] YF_FullCloud - Breakup 2025
Loading: 20250619_214200_08_2507_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[393/578] YF_FullCloud - Breakup 2020
Loading: 20200606_210015_1018_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[394/578] YF_FullCloud - Breakup 2020
Loading: 20200419_173547_1050_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[395/578] YF_FullCloud - Breakup 2021
Loading: 20210704_162222_104a_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[396/578] YF_FullCloud - Breakup 2022
Loading: 20220606_205349_33_248f_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[397/578] YF_FullCloud - Breakup 2025
Loading: 20250613_215003_48_253b_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[398/578] YF_FullCloud - Breakup 2022
Loading: 20220620_202415_07_241e_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[399/578] YF_FullCloud - Breakup 2019
Loading: 20190427_204840_1004_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[400/578] YF_FullCloud - Breakup 2019
Loading: 20190615_205429_1004_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[401/578] YF_FullCloud - Breakup 2019
Loading: 20190515_205951_1009_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[402/578] YF_FullCloud - Breakup 2020
Loading: 20200417_192907_71_106c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[403/578] YF_FullCloud - Breakup 2022
Loading: 20220605_202358_91_2460_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[404/578] YF_FullCloud - Breakup 2019
Loading: 20190430_205736_101e_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[405/578] YF_FullCloud - Breakup 2021
Loading: 20210420_204744_1034_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[406/578] YF_FullCloud - Breakup 2023
Loading: 20230428_211735_22_2426_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[407/578] YF_FullCloud - Breakup 2024
Loading: 20240607_213227_07_24bd_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[408/578] YF_FullCloud - Breakup 2020
Loading: 20200415_211719_70_105c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[409/578] YF_FullCloud - Breakup 2021
Loading: 20210617_211715_07_2407_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[410/578] YF_FullCloud - Breakup 2024
Loading: 20240713_203230_42_24a8_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[411/578] YF_FullCloud - Breakup 2025
Loading: 20250604_213752_51_251c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[412/578] YF_FullCloud - Breakup 2020
Loading: 20200617_210205_103e_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[413/578] YF_FullCloud - Breakup 2021
Loading: 20210418_203913_0e0f_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[414/578] YF_FullCloud - Breakup 2023
Loading: 20230514_210704_89_227a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[415/578] YF_FullCloud - Breakup 2021
Loading: 20210418_163501_104e_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[416/578] YF_FullCloud - Breakup 2020
Loading: 20200516_212049_17_105d_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[417/578] YF_FullCloud - Breakup 2020
Loading: 20200602_205538_1029_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[418/578] YF_FullCloud - Breakup 2025
Loading: 20250510_214552_46_2540_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[419/578] YF_FullCloud - Breakup 2025
Loading: 20250710_214701_85_24e9_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[420/578] YF_FullCloud - Breakup 2020
Loading: 20200510_173513_0f32_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[421/578] YF_FullCloud - Breakup 2021
Loading: 20210526_211257_49_2403_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[422/578] YF_FullCloud - Breakup 2025
Loading: 20250610_210552_95_2417_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[423/578] YF_FullCloud - Breakup 2024
Loading: 20240614_213443_72_2470_ortho_analytic_4b_sr.tif
Going back to image 422: 20250610_210552_95_2417_ortho_analytic_4b_sr.tif

[422/578] YF_FullCloud - Breakup 2025
Loading: 20250610_210552_95_2417_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[423/578] YF_FullCloud - Breakup 2024
Loading: 20240614_213443_72_2470_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[424/578] YF_FullCloud - Breakup 2024
Loading: 20240504_202630_38_24c0_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[425/578] YF_FullCloud - Breakup 2022
Loading: 20220423_215551_75_1057_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[426/578] YF_FullCloud - Breakup 2022
Loading: 20220528_205541_24_247a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[427/578] YF_FullCloud - Breakup 2023
Loading: 20230507_202612_73_24cf_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[428/578] YF_FullCloud - Breakup 2023
Loading: 20230628_210017_57_2481_ortho_analytic_4b_sr.tif
Going back to image 427: 20230507_202612_73_24cf_ortho_analytic_4b_sr.tif

[427/578] YF_FullCloud - Breakup 2023
Loading: 20230507_202612_73_24cf_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[428/578] YF_FullCloud - Breakup 2023
Loading: 20230628_210017_57_2481_ortho_analytic_4b_sr.tif
Going back to image 427: 20230507_202612_73_24cf_ortho_analytic_4b_sr.tif

[427/578] YF_FullCloud - Breakup 2023
Loading: 20230507_202612_73_24cf_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[428/578] YF_FullCloud - Breakup 2023
Loading: 20230628_210017_57_2481_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[429/578] YF_FullCloud - Breakup 2019
Loading: 20190618_205042_1024_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[430/578] YF_FullCloud - Breakup 2019
Loading: 20190520_204646_101f_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[431/578] YF_FullCloud - Breakup 2025
Loading: 20250626_214739_46_2535_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[432/578] YF_FullCloud - Breakup 2021
Loading: 20210704_162225_104a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[433/578] YF_FullCloud - Breakup 2024
Loading: 20240701_212151_22_24fd_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[434/578] YF_FullCloud - Breakup 2021
Loading: 20210709_162213_0f21_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[435/578] YF_FullCloud - Breakup 2020
Loading: 20200422_212325_70_1064_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[436/578] YF_FullCloud - Breakup 2021
Loading: 20210430_190836_30_1067_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[437/578] YF_FullCloud - Breakup 2021
Loading: 20210712_214329_70_105a_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[438/578] YF_FullCloud - Breakup 2021
Loading: 20210602_213937_66_105e_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[439/578] YF_FullCloud - Breakup 2019
Loading: 20190710_204554_0e2f_ortho_analytic_3b.tif
Labeled as: very_cloudy

[440/578] YF_FullCloud - Breakup 2021
Loading: 20210529_210725_47_105c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[441/578] YF_FullCloud - Breakup 2021
Loading: 20210711_162235_0f2b_ortho_analytic_3b.tif
Labeled as: very_cloudy

[442/578] YF_FullCloud - Breakup 2020
Loading: 20200516_205706_0f22_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[443/578] YF_FullCloud - Breakup 2019
Loading: 20190427_204928_1006_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[444/578] YF_FullCloud - Breakup 2020
Loading: 20200602_173138_0f3d_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[445/578] YF_FullCloud - Breakup 2023
Loading: 20230708_202555_81_24d0_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[446/578] YF_FullCloud - Breakup 2019
Loading: 20190606_205610_1012_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[447/578] YF_FullCloud - Breakup 2020
Loading: 20200529_210107_1039_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[448/578] YF_FullCloud - Breakup 2021
Loading: 20210710_203107_93_2206_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[449/578] YF_FullCloud - Breakup 2023
Loading: 20230611_201954_07_24ca_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[450/578] YF_FullCloud - Breakup 2021
Loading: 20210415_204749_1003_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[451/578] YF_FullCloud - Breakup 2020
Loading: 20200516_205709_0f22_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[452/578] YF_FullCloud - Breakup 2020
Loading: 20200517_205302_103c_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[453/578] YF_FullCloud - Breakup 2019
Loading: 20190509_205744_1006_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[454/578] YF_FullCloud - Breakup 2020
Loading: 20200521_172614_104e_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[455/578] YF_FullCloud - Breakup 2025
Loading: 20250517_213230_49_2515_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[456/578] YF_FullCloud - Breakup 2022
Loading: 20220428_210719_70_2407_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[457/578] YF_FullCloud - Breakup 2024
Loading: 20240504_203532_31_24a8_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[458/578] YF_FullCloud - Breakup 2020
Loading: 20200607_205722_1013_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[459/578] YF_FullCloud - Breakup 2020
Loading: 20200629_171926_0f36_ortho_analytic_3b.tif
Labeled as: very_cloudy

[460/578] YF_FullCloud - Breakup 2019
Loading: 20190515_205532_0f3f_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[461/578] YF_FullCloud - Breakup 2020
Loading: 20200611_172506_0f2b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[462/578] YF_FullCloud - Breakup 2024
Loading: 20240619_212405_21_24ed_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[463/578] YF_FullCloud - Breakup 2022
Loading: 20220527_204401_49_2233_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[464/578] YF_FullCloud - Breakup 2023
Loading: 20230417_210038_28_248f_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[465/578] YF_FullCloud - Breakup 2025
Loading: 20250714_214402_63_250a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[466/578] YF_FullCloud - Breakup 2020
Loading: 20200615_172341_1020_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[467/578] YF_FullCloud - Breakup 2019
Loading: 20190614_205423_1039_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[468/578] YF_FullCloud - Breakup 2022
Loading: 20220712_205803_16_2481_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[469/578] YF_FullCloud - Breakup 2022
Loading: 20220508_203528_56_2276_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[470/578] YF_FullCloud - Breakup 2024
Loading: 20240629_212306_94_24f4_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[471/578] YF_FullCloud - Breakup 2021
Loading: 20210616_162400_1048_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[472/578] YF_FullCloud - Breakup 2023
Loading: 20230511_201547_38_2430_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[473/578] YF_FullCloud - Breakup 2022
Loading: 20220419_202452_80_2429_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[474/578] YF_FullCloud - Breakup 2020
Loading: 20200710_192350_43_106a_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[475/578] YF_FullCloud - Breakup 2020
Loading: 20200609_172044_0f21_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[476/578] YF_FullCloud - Breakup 2021
Loading: 20210705_205605_1010_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[477/578] YF_FullCloud - Breakup 2020
Loading: 20200511_212736_36_1059_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[478/578] YF_FullCloud - Breakup 2020
Loading: 20200701_205643_1006_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[479/578] YF_FullCloud - Breakup 2021
Loading: 20210608_191301_79_106d_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[480/578] YF_FullCloud - Breakup 2019
Loading: 20190428_205719_0f15_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[481/578] YF_FullCloud - Freezeup 2025
Loading: 20251012_214350_35_251f_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[482/578] YF_FullCloud - Freezeup 2021
Loading: 20211019_205250_1010_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[483/578] YF_FullCloud - Freezeup 2022
Loading: 20221016_201953_69_2459_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[484/578] YF_FullCloud - Freezeup 2024
Loading: 20241007_212906_70_24b7_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[485/578] YF_FullCloud - Freezeup 2023
Loading: 20231016_202621_07_24a9_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[486/578] YF_FullCloud - Freezeup 2020
Loading: 20201002_205217_1039_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[487/578] YF_FullCloud - Freezeup 2022
Loading: 20221005_201358_08_2451_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[488/578] YF_FullCloud - Freezeup 2024
Loading: 20241003_213023_64_24c6_ortho_analytic_4b_sr.tif
Going back to image 487: 20221005_201358_08_2451_ortho_analytic_4b_sr.tif

[487/578] YF_FullCloud - Freezeup 2022
Loading: 20221005_201358_08_2451_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[488/578] YF_FullCloud - Freezeup 2024
Loading: 20241003_213023_64_24c6_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[489/578] YF_FullCloud - Freezeup 2020
Loading: 20201017_203549_11_2235_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[490/578] YF_FullCloud - Freezeup 2021
Loading: 20211002_205549_1005_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[491/578] YF_FullCloud - Freezeup 2019
Loading: 20191016_210718_00_105a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[492/578] YF_FullCloud - Freezeup 2025
Loading: 20251013_213944_35_2526_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[493/578] YF_FullCloud - Freezeup 2019
Loading: 20191002_194708_99_106f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[494/578] YF_FullCloud - Freezeup 2023
Loading: 20231016_210710_02_2495_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[495/578] YF_FullCloud - Freezeup 2023
Loading: 20231021_201705_54_2455_ortho_analytic_4b_sr.tif
Going back to image 494: 20231016_210710_02_2495_ortho_analytic_4b_sr.tif

[494/578] YF_FullCloud - Freezeup 2023
Loading: 20231016_210710_02_2495_ortho_analytic_4b_sr.tif
Going back to image 493: 20191002_194708_99_106f_ortho_analytic_4b_sr.tif

[493/578] YF_FullCloud - Freezeup 2019
Loading: 20191002_194708_99_106f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[494/578] YF_FullCloud - Freezeup 2023
Loading: 20231016_210710_02_2495_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[495/578] YF_FullCloud - Freezeup 2023
Loading: 20231021_201705_54_2455_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[496/578] YF_FullCloud - Freezeup 2020
Loading: 20201017_205542_1001_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[497/578] YF_FullCloud - Freezeup 2021
Loading: 20211023_202751_39_2431_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[498/578] YF_FullCloud - Freezeup 2024
Loading: 20241023_204503_38_24ca_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[499/578] YF_FullCloud - Freezeup 2019
Loading: 20191020_211748_32_1060_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[500/578] YF_FullCloud - Freezeup 2021
Loading: 20211020_202044_26_242b_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[501/578] YF_FullCloud - Freezeup 2025
Loading: 20251016_214540_18_2539_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[502/578] YF_FullCloud - Freezeup 2020
Loading: 20201003_205200_1004_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[503/578] YF_FullCloud - Freezeup 2021
Loading: 20211019_203246_21_2460_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[504/578] YF_FullCloud - Freezeup 2023
Loading: 20231015_203204_58_24bb_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[505/578] YF_FullCloud - Freezeup 2021
Loading: 20211019_203420_51_2465_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[506/578] YF_FullCloud - Freezeup 2022
Loading: 20221011_204956_33_2486_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[507/578] YF_FullCloud - Freezeup 2021
Loading: 20211019_210426_1026_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[508/578] YF_FullCloud - Freezeup 2022
Loading: 20221009_210610_53_241c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[509/578] YF_FullCloud - Freezeup 2021
Loading: 20211019_203243_91_2460_ortho_analytic_4b_sr.tif
Going back to image 508: 20221009_210610_53_241c_ortho_analytic_4b_sr.tif

[508/578] YF_FullCloud - Freezeup 2022
Loading: 20221009_210610_53_241c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[509/578] YF_FullCloud - Freezeup 2021
Loading: 20211019_203243_91_2460_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[510/578] YF_FullCloud - Freezeup 2020
Loading: 20201005_213012_22_105a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[511/578] YF_FullCloud - Freezeup 2025
Loading: 20251007_214014_21_24db_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[512/578] YF_FullCloud - Freezeup 2022
Loading: 20221012_205052_68_2492_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[513/578] YF_FullCloud - Freezeup 2022
Loading: 20221004_210502_63_2274_ortho_analytic_4b_sr.tif
Going back to image 512: 20221012_205052_68_2492_ortho_analytic_4b_sr.tif

[512/578] YF_FullCloud - Freezeup 2022
Loading: 20221012_205052_68_2492_ortho_analytic_4b_sr.tif
Going back to image 511: 20251007_214014_21_24db_ortho_analytic_4b_sr.tif

[511/578] YF_FullCloud - Freezeup 2025
Loading: 20251007_214014_21_24db_ortho_analytic_4b_sr.tif
Going back to image 510: 20201005_213012_22_105a_ortho_analytic_4b_sr.tif

[510/578] YF_FullCloud - Freezeup 2020
Loading: 20201005_213012_22_105a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[511/578] YF_FullCloud - Freezeup 2025
Loading: 20251007_214014_21_24db_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[512/578] YF_FullCloud - Freezeup 2022
Loading: 20221012_205052_68_2492_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[513/578] YF_FullCloud - Freezeup 2022
Loading: 20221004_210502_63_2274_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[514/578] YF_FullCloud - Freezeup 2020
Loading: 20201013_210053_1005_ortho_analytic_3b.tif
Going back to image 513: 20221004_210502_63_2274_ortho_analytic_4b_sr.tif

[513/578] YF_FullCloud - Freezeup 2022
Loading: 20221004_210502_63_2274_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[514/578] YF_FullCloud - Freezeup 2020
Loading: 20201013_210053_1005_ortho_analytic_3b.tif
Labeled as: very_cloudy

[515/578] YF_FullCloud - Freezeup 2024
Loading: 20241004_212608_52_251a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[516/578] YF_FullCloud - Freezeup 2024
Loading: 20241020_212536_33_251c_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[517/578] YF_FullCloud - Freezeup 2021
Loading: 20211012_185330_74_1067_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[518/578] YF_FullCloud - Freezeup 2024
Loading: 20241022_213117_20_2500_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[519/578] YF_FullCloud - Freezeup 2024
Loading: 20241022_214009_58_2514_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[520/578] YF_FullCloud - Freezeup 2020
Loading: 20201012_213136_05_105a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[521/578] YF_FullCloud - Freezeup 2020
Loading: 20201011_205533_1035_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[522/578] YF_FullCloud - Freezeup 2020
Loading: 20201002_205216_1039_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[523/578] YF_FullCloud - Freezeup 2020
Loading: 20201020_210006_0f22_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[524/578] YF_FullCloud - Freezeup 2022
Loading: 20221007_202018_71_241a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[525/578] YF_FullCloud - Freezeup 2023
Loading: 20231022_202628_84_2459_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[526/578] YF_FullCloud - Freezeup 2023
Loading: 20231006_205913_40_247f_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[527/578] YF_FullCloud - Freezeup 2024
Loading: 20241001_203847_67_24c4_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[528/578] YF_FullCloud - Freezeup 2020
Loading: 20201013_203021_91_2277_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[529/578] YF_FullCloud - Freezeup 2020
Loading: 20201026_205645_1040_ortho_analytic_3b.tif
Labeled as: very_cloudy

[530/578] YF_FullCloud - Freezeup 2020
Loading: 20201023_204257_0e3a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[531/578] YF_FullCloud - Freezeup 2025
Loading: 20251016_214542_49_2539_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[532/578] YF_FullCloud - Freezeup 2021
Loading: 20211020_214425_67_1064_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[533/578] YF_FullCloud - Freezeup 2022
Loading: 20221016_205357_92_2475_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[534/578] YF_FullCloud - Freezeup 2024
Loading: 20241027_213204_90_24d9_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[535/578] YF_FullCloud - Freezeup 2020
Loading: 20201026_210522_0f28_ortho_analytic_3b.tif
Labeled as: very_cloudy

[536/578] YF_FullCloud - Freezeup 2019
Loading: 20191024_205758_103b_ortho_analytic_3b.tif
Labeled as: very_cloudy

[537/578] YF_FullCloud - Freezeup 2024
Loading: 20241023_212635_46_24f5_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[538/578] YF_FullCloud - Freezeup 2022
Loading: 20221021_201828_97_2430_ortho_analytic_4b_sr.tif
Going back to image 537: 20241023_212635_46_24f5_ortho_analytic_4b_sr.tif

[537/578] YF_FullCloud - Freezeup 2024
Loading: 20241023_212635_46_24f5_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[538/578] YF_FullCloud - Freezeup 2022
Loading: 20221021_201828_97_2430_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[539/578] YF_FullCloud - Freezeup 2021
Loading: 20211021_205431_1012_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[540/578] YF_FullCloud - Freezeup 2020
Loading: 20201013_205254_1003_ortho_analytic_3b.tif
Labeled as: very_cloudy

[541/578] YF_FullCloud - Freezeup 2023
Loading: 20231024_210643_77_248b_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[542/578] YF_FullCloud - Freezeup 2024
Loading: 20241007_204725_42_24af_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[543/578] YF_FullCloud - Freezeup 2022
Loading: 20221026_205806_32_2483_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[544/578] YF_FullCloud - Freezeup 2025
Loading: 20251012_213702_72_24f0_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[545/578] YF_FullCloud - Freezeup 2020
Loading: 20201001_213201_52_1061_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[546/578] YF_FullCloud - Freezeup 2019
Loading: 20191022_205410_1025_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[547/578] YF_FullCloud - Freezeup 2019
Loading: 20191011_194229_15_1065_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[548/578] YF_FullCloud - Freezeup 2021
Loading: 20211002_205546_1005_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[549/578] YF_FullCloud - Freezeup 2019
Loading: 20191017_212000_33_105c_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[550/578] YF_FullCloud - Freezeup 2021
Loading: 20211012_204742_1010_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[551/578] YF_FullCloud - Freezeup 2023
Loading: 20231004_202956_06_24bf_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[552/578] YF_FullCloud - Freezeup 2023
Loading: 20231018_210546_95_247a_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[553/578] YF_FullCloud - Freezeup 2019
Loading: 20191011_205938_0f25_ortho_analytic_3b.tif
Labeled as: very_cloudy

[554/578] YF_FullCloud - Freezeup 2021
Loading: 20211008_203335_16_2440_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[555/578] YF_FullCloud - Freezeup 2021
Loading: 20211022_205039_1008_ortho_analytic_3b.tif
Labeled as: very_cloudy

[556/578] YF_FullCloud - Freezeup 2022
Loading: 20221013_204947_44_2488_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[557/578] YF_FullCloud - Freezeup 2025
Loading: 20251012_214401_41_251f_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[558/578] YF_FullCloud - Freezeup 2019
Loading: 20191016_211526_22_105d_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[559/578] YF_FullCloud - Freezeup 2021
Loading: 20211002_211714_79_2414_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[560/578] YF_FullCloud - Freezeup 2021
Loading: 20211024_205141_103c_ortho_analytic_3b.tif
Labeled as: very_cloudy

[561/578] YF_FullCloud - Freezeup 2024
Loading: 20241017_204148_68_24b0_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[562/578] YF_FullCloud - Freezeup 2025
Loading: 20251011_215117_72_2547_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[563/578] YF_FullCloud - Freezeup 2019
Loading: 20191015_205434_103d_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[564/578] YF_FullCloud - Freezeup 2024
Loading: 20241014_203828_22_24b3_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[565/578] YF_FullCloud - Freezeup 2020
Loading: 20201001_211908_14_2402_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[566/578] YF_FullCloud - Freezeup 2022
Loading: 20221026_205808_68_2483_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[567/578] YF_FullCloud - Freezeup 2024
Loading: 20241018_203948_96_24cf_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[568/578] YF_FullCloud - Freezeup 2024
Loading: 20241004_203636_91_24c4_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[569/578] YF_FullCloud - Freezeup 2019
Loading: 20191014_204615_0e20_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[570/578] YF_FullCloud - Freezeup 2023
Loading: 20231005_202113_01_2431_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[571/578] YF_FullCloud - Freezeup 2024
Loading: 20241020_213146_18_251a_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[572/578] YF_FullCloud - Freezeup 2020
Loading: 20201009_210047_1001_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[573/578] YF_FullCloud - Freezeup 2024
Loading: 20241002_204154_32_24a8_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[574/578] YF_FullCloud - Freezeup 2020
Loading: 20201014_210008_1034_ortho_analytic_3b.tif
Labeled as: very_cloudy

[575/578] YF_FullCloud - Freezeup 2020
Loading: 20201017_205547_1001_ortho_analytic_4b_sr.tif
Labeled as: no_clouds

[576/578] YF_FullCloud - Freezeup 2023
Loading: 20231003_211238_70_247b_ortho_analytic_4b_sr.tif
Labeled as: cloudy

[577/578] YF_FullCloud - Freezeup 2025
Loading: 20251009_215155_69_2546_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

[578/578] YF_FullCloud - Freezeup 2024
Loading: 20241016_212051_45_24e1_ortho_analytic_4b_sr.tif
Labeled as: very_cloudy

============================================================
LABELING SUMMARY
============================================================
YKD_FullCloud: 181 images labeled
NS_FullCloud: 200 images labeled
YF_FullCloud: 197 images labeled

Total labeled: 578 / 578
============================================================